# LLM Text Preprocessing Foundations — Embeddings

Based on Chapter 2 of *Build a Large Language Model (From Scratch)* by Sebastian Raschka.

In [1]:
%pip install torch tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from importlib.metadata import version
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.10.0
tiktoken version: 0.12.0


## Why Text Preprocessing Matters for LLMs

Before a Large Language Model can do anything useful, raw text must be transformed into numbers. Neural networks cannot process words directly — they only understand tensors of floating point numbers. This chapter covers exactly that transformation pipeline:

1. Raw text → tokens (tokenization)
2. Tokens → integer IDs (vocabulary mapping)
3. Integer IDs → dense vectors (embeddings)

Each step is essential. Without tokenization there is no consistent unit of meaning. Without vocabulary mapping there are no numbers to feed the network. Without embeddings there is no way for the model to learn semantic relationships between words.

This pipeline is not just a preprocessing detail — it is the foundation on which the entire model is built. The quality of this pipeline directly affects what the model can learn.

In [3]:
import os

if not os.path.exists("the-verdict.txt"):
    import requests
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
    response = requests.get(url, timeout=30)
    with open("the-verdict.txt", "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:99])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## 2.2 Tokenizing Text

Breaking raw text into individual tokens (words and punctuation).

In [4]:
import re

text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [5]:
preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print("Total tokens:", len(preprocessed))
print("First 30 tokens:", preprocessed[:30])

Total tokens: 4690
First 30 tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Why Tokenization Is a Design Decision

The way we split text into tokens has a direct impact on the vocabulary size and on what the model considers a single unit of meaning.

A simple whitespace split would treat `"running"` and `"run"` as completely different tokens, doubling the vocabulary size unnecessarily. A character-level split would make sequences very long and harder to learn from.

The regex-based tokenizer used here strikes a balance: it keeps words intact but separates punctuation as its own token. This means the model can learn that `"Hello,"` and `"Hello"` are closely related, because they share the token `"Hello"`.

In production LLMs like GPT, Byte Pair Encoding (BPE) is used instead, which learns the optimal tokenization from data. But the principle is the same: every tokenization choice is a trade-off between vocabulary size, sequence length, and the ability to generalize.

## 2.3 Converting Tokens into Token IDs

In [6]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Vocabulary size:", vocab_size)

vocab = {token: integer for integer, token in enumerate(all_words)}

# Show first 10 entries
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 10:
        break

Vocabulary size: 1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


In [7]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!\"()\'])', r'\1', text)
        return text

tokenizer = SimpleTokenizerV1(vocab)
text = """\"It's the last he painted, you know,\" 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print("Encoded:", ids)
print("Decoded:", tokenizer.decode(ids))

Encoded: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
Decoded: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


## 2.4 Adding Special Context Tokens

In [8]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print("New vocabulary size:", len(vocab))

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!\"()\'])', r'\1', text)
        return text

tokenizer = SimpleTokenizerV2(vocab)
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print("Text:", text)
print("Encoded:", tokenizer.encode(text))
print("Decoded:", tokenizer.decode(tokenizer.encode(text)))

New vocabulary size: 1132
Text: Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
Encoded: [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
Decoded: <|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## 2.5 Byte Pair Encoding with tiktoken

In [9]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Encoded:", integers)
print("Decoded:", tokenizer.decode(integers))

Encoded: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Decoded: Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


## Why BPE (Byte Pair Encoding) Is Better Than Simple Tokenization

The simple tokenizer we built has a critical flaw: if it encounters a word not in its vocabulary, it replaces it with `<|unk|>`, losing all information.

BPE solves this by building a vocabulary from subword units. A word like `"someunknownPlace"` would be split into known subwords like `["some", "unknown", "Place"]` instead of replaced with a single unknown token. This means:

- The vocabulary stays compact (GPT-2 uses 50,257 tokens)
- No word is truly unknown — it can always be broken into subwords or characters
- The model can generalize to new words it has never seen during training

This is why modern LLMs can handle technical jargon, code, foreign words, and even typos — the BPE tokenizer gracefully degrades to smaller units rather than giving up entirely.

## 2.6 Data Sampling with a Sliding Window

In [10]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print("Total tokens in text:", len(enc_text))

enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"\nInput  x: {x}")
print(f"Target y: {y}")

print("\nNext-token prediction pairs:")
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"  {tokenizer.decode(context)} ----> {tokenizer.decode([desired])}")

Total tokens in text: 5145

Input  x: [290, 4920, 2241, 287]
Target y: [4920, 2241, 287, 257]

Next-token prediction pairs:
   and ---->  established
   and established ---->  himself
   and established himself ---->  in
   and established himself in ---->  a


In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size,
                            shuffle=shuffle, drop_last=drop_last,
                            num_workers=num_workers)
    return dataloader

# Default example from the book
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print("First batch:", first_batch)
second_batch = next(data_iter)
print("Second batch:", second_batch)

First batch: [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
Second batch: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


## Experiment: How max_length and stride Affect the Number of Samples

In this experiment we change `max_length` and `stride` to observe how many training samples are generated and why overlap (stride < max_length) is useful.

In [12]:
import tiktoken
tokenizer_exp = tiktoken.get_encoding("gpt2")

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text_exp = f.read()

total_tokens = len(tokenizer_exp.encode(raw_text_exp))
print(f"Total tokens in text: {total_tokens}\n")

configs = [
    {"max_length": 4,   "stride": 1,   "label": "Short window, stride=1 (heavy overlap)"},
    {"max_length": 4,   "stride": 4,   "label": "Short window, stride=4 (no overlap)"},
    {"max_length": 64,  "stride": 32,  "label": "Medium window, stride=32 (50% overlap)"},
    {"max_length": 64,  "stride": 64,  "label": "Medium window, stride=64 (no overlap)"},
    {"max_length": 256, "stride": 128, "label": "Large window, stride=128 (50% overlap)"},
    {"max_length": 256, "stride": 256, "label": "Large window, stride=256 (no overlap)"},
]

print(f"{'Config':<50} {'Samples':>8}")
print("-" * 60)
for cfg in configs:
    dl = create_dataloader_v1(
        raw_text_exp,
        batch_size=1,
        max_length=cfg["max_length"],
        stride=cfg["stride"],
        shuffle=False,
        drop_last=False
    )
    print(f"{cfg['label']:<50} {len(dl):>8}")

Total tokens in text: 5145

Config                                              Samples
------------------------------------------------------------
Short window, stride=1 (heavy overlap)                 5141
Short window, stride=4 (no overlap)                    1286
Medium window, stride=32 (50% overlap)                  159
Medium window, stride=64 (no overlap)                    80
Large window, stride=128 (50% overlap)                   39
Large window, stride=256 (no overlap)                    20


### Experiment Results and Analysis

From the experiment we can observe two clear patterns:

**1. Smaller stride = more samples**
When stride is smaller than max_length the sliding window moves slowly through the text, creating many overlapping samples. When stride equals max_length the window jumps forward without overlap, creating fewer non-redundant samples.

**2. Why overlap is useful during training**
Each training sample teaches the model to predict the next token given a context window. With overlap, the same token appears in multiple samples but always with a different surrounding context. This means the model sees many different ways the same word can be predicted, which improves generalization.

Without overlap (stride = max_length) the model only sees each token once in training, which can lead to underfitting, especially for a short text like this one.

In practice GPT-2 was trained with stride = max_length because the training dataset was enormous and overlap was not necessary. For smaller datasets, overlap is a simple way to artificially increase the number of training examples.

## 2.7 Token Embedding and Positional Embedding

In [13]:
# Simple embedding example
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print("Embedding weights:\n", embedding_layer.weight)
print("\nEmbedding for token 3:", embedding_layer(torch.tensor([3])))
print("\nEmbeddings for all input_ids:\n", embedding_layer(input_ids))

Embedding weights:
 Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

Embedding for token 3: tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

Embeddings for all input_ids:
 tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [14]:

vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Token IDs shape:", inputs.shape)

token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)

# Positional embeddings
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print("Positional embeddings shape:", pos_embeddings.shape)

# Final input embeddings
input_embeddings = token_embeddings + pos_embeddings
print("Final input embeddings shape:", input_embeddings.shape)

Token IDs shape: torch.Size([8, 4])
Token embeddings shape: torch.Size([8, 4, 256])
Positional embeddings shape: torch.Size([4, 256])
Final input embeddings shape: torch.Size([8, 4, 256])


## Why Embeddings Encode Meaning — Connection to Neural Network Concepts

**Why do embeddings encode meaning?**

An embedding is a lookup table: each token ID maps to a row of learnable weights. At the start of training these weights are random. As the model trains, it updates these weights using backpropagation to minimize the prediction error.

Because words that appear in similar contexts (like "king" and "queen") produce similar prediction errors when swapped, the optimizer nudges their embedding vectors to be close together in the vector space. Over millions of updates, the geometry of the embedding space comes to reflect semantic relationships: similar meanings cluster together, and directions in the space encode relationships like gender, tense, or topic.

**Connection to neural network concepts:**

- `torch.nn.Embedding` is simply a linear layer without a bias, where the input is a one-hot vector. It is mathematically equivalent to a matrix multiplication but implemented as an efficient index lookup.
- The weights of the embedding layer are learned through backpropagation exactly like any other weight in the network.
- Positional embeddings are added to token embeddings so the model knows the order of tokens. Without them, a transformer would treat every permutation of the same words as identical — it has no recurrent structure and no built-in sense of sequence.

In summary, embeddings are the bridge between discrete symbolic language and the continuous vector space where neural networks do their computation. The meaning they encode is not programmed in — it emerges from the structure of the data through gradient descent.

## Summary

This notebook covered the complete text preprocessing pipeline for LLMs:

1. **Tokenization** — splitting raw text into discrete units using regex or BPE
2. **Vocabulary mapping** — converting tokens to integer IDs
3. **Special tokens** — handling unknown words and text boundaries
4. **Sliding window sampling** — creating input/target pairs for next-token prediction training
5. **Token embeddings** — mapping integer IDs to dense learnable vectors
6. **Positional embeddings** — adding sequence order information to the embeddings

The final output `input_embeddings` with shape `(batch_size, max_length, embedding_dim)` is exactly what gets fed into the attention layers of a transformer.